# RAG Pipeline - Complete Implementation

## Expert Knowledge Worker for Legal Documents

This notebook combines all RAG (Retrieval Augmented Generation) pipeline components:
- **Day 1**: Basic RAG with keyword matching
- **Day 2**: Document chunking and vector embeddings with Chroma
- **Day 3**: LangChain-based RAG with retriever and LLM
- **Day 4**: Evaluation of the RAG pipeline

Knowledge Base: Indian Constitution (COI), Indian Penal Code (IPC), Bharatiya Nyaya Sanhita (BNS)

---
## Part 1: Setup and Imports

In [18]:
import os
import tiktoken
import numpy as np

from dotenv import load_dotenv
import gradio as gr

from sklearn.manifold import TSNE
import plotly.graph_objects as go

from app import format_context
from implementation.ingest import fetch_documents, create_chunks, create_embeddings
from implementation.answer import answer_question as impl_answer_question, fetch_context
from evaluation.test import load_tests
from evaluation.eval import evaluate_retrieval, evaluate_answer, evaluate_all_retrieval, evaluate_all_answers
from evaluator import get_color, run_retrieval_evaluation, run_answer_evaluation
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [19]:
# Configuration
load_dotenv(override=True)

MODEL = "llama3.2:latest"
DB_NAME = "vector_db"

print(f"Model: {MODEL}")
print(f"Vector DB: {DB_NAME}")
print("Notebook is wired to app.py, implementation/*.py, evaluator.py, and evaluation/*.py")

Model: llama3.2:latest
Vector DB: vector_db
Notebook is wired to app.py, implementation/*.py, evaluator.py, and evaluation/*.py


---
## Part 2: Load Knowledge Base (Day 1 Approach)

In [20]:
# Load documents using implementation.ingest
documents = fetch_documents()

print(f"Loaded {len(documents)} documents via implementation.ingest.fetch_documents()")
if documents:
    print("Sample metadata:", documents[0].metadata)

Loaded 3 documents via implementation.ingest.fetch_documents()
Sample metadata: {'source': '/home/lucifer049/Documents/imp/ML+Python/resume ones/rag-legal/knowledge-base/coi.txt', 'doc_type': 'coi'}


In [21]:
# Tokens in loaded corpus
entire_knowledge_base = "\n\n".join(doc.page_content for doc in documents)
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(entire_knowledge_base)

print(f"Total characters: {len(entire_knowledge_base):,}")
print(f"Approx token count: {len(tokens):,}")

Total characters: 1,741,672
Approx token count: 413,711


---
## Part 3: Document Chunking & Vector Store (Day 2)

In [22]:
# Count tokens in the loaded documents
knowledge = [doc.page_content for doc in documents]
entire_knowledge_base = "\n\n".join(knowledge)

encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(entire_knowledge_base)

print(f"Total characters: {len(entire_knowledge_base):,}")
print(f"Approx token count: {len(tokens):,}")

Total characters: 1,741,672
Approx token count: 413,711


In [23]:
# Split into chunks using implementation.ingest
chunks = create_chunks(documents)
print(f"Created {len(chunks)} chunks via implementation.ingest.create_chunks()")

Created 5740 chunks via implementation.ingest.create_chunks()


In [24]:
# Create embeddings/vector DB using implementation.ingest
vectorstore = create_embeddings(chunks)
print(f"Vector store ready with {vectorstore._collection.count()} vectors")

There are 5,740 vectors with 384 dimensions in the vector store
Vector store ready with 5740 vectors


In [25]:
# Create/Load vector store with HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Check if vector DB exists
if os.path.exists(DB_NAME):
    print(f"Loading existing vector store from {DB_NAME}")
    vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
else:
    print(f"Creating new vector store at {DB_NAME}")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME
    )

print(f"Vector store ready with {vectorstore._collection.count()} vectors")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5846.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading existing vector store from vector_db
Vector store ready with 5740 vectors


In [26]:
# Visualize vectors in 2D (optional)
def visualize_vectors_2d(vectorstore, sample_size=500):
    """Visualize vector embeddings using t-SNE."""
    collection = vectorstore._collection
    result = collection.get(include=['embeddings', 'documents', 'metadatas'])
    
    vectors = np.array(result['embeddings'][:sample_size])
    documents = result['documents'][:sample_size]
    doc_types = [m.get('doc_type', 'unknown') for m in result['metadatas'][:sample_size]]
    
    # Reduce to 2D
    tsne = TSNE(n_components=2, random_state=42)
    reduced = tsne.fit_transform(vectors)
    
    # Color mapping
    colors = {'coi': 'blue', 'ipc': 'red', 'bns': 'green'}
    point_colors = [colors.get(dt, 'gray') for dt in doc_types]
    
    fig = go.Figure(data=[go.Scatter(
        x=reduced[:, 0],
        y=reduced[:, 1],
        mode='markers',
        marker=dict(size=5, color=point_colors, opacity=0.7),
        text=[d[:100] + '...' for d in documents],
        hoverinfo='text'
    )])
    
    fig.update_layout(title='Vector Embeddings Visualization (2D t-SNE)')
    fig.show()

# Uncomment to visualize
visualize_vectors_2d(vectorstore)

---
## Part 4: RAG Pipeline with LangChain (Day 3)

In [27]:
# Use implementation.answer retrieval directly
sample_docs = fetch_context("What is Article 21 of the Indian Constitution?")
print(f"Retrieved {len(sample_docs)} docs via implementation.answer.fetch_context()")

Retrieved 10 docs via implementation.answer.fetch_context()


In [28]:
# Wrap implementation.answer for notebook and Gradio usage
def answer_question(question: str, history=None):
    history = history or []
    answer, _ = impl_answer_question(question, history)
    return answer

def answer_with_context(question: str, history=None):
    history = history or []
    answer, context_docs = impl_answer_question(question, history)
    return answer, format_context(context_docs)

In [29]:
# Test the RAG pipeline through implementation.answer
test_questions = [
    "What is Article 21 of the Indian Constitution?",
    "What is the definition of 'child' under BNS 2023?",
    "What is Section 302 of IPC about?"
]

for q in test_questions:
    print(f"\nQ: {q}")
    answer, context_html = answer_with_context(q, [])
    print(f"A: {answer[:300]}...")
    print(f"Context preview length: {len(context_html)}")
    print("-" * 50)


Q: What is Article 21 of the Indian Constitution?
A: Article 21 of the Indian Constitution is a fundamental right that guarantees the right to life and personal liberty. It states:

"Nothing in this Part shall be taken to abridge the right to freedom of speech and expression; and the State shall not make any law which abridges the right of free speech...
Context preview length: 5620
--------------------------------------------------

Q: What is the definition of 'child' under BNS 2023?
A: I don't have the specific information on the definition of 'child' under the Bharatiya Nyaya Sanhita (BNS) 2023. My training data only goes up to a certain point, and I may not have access to the latest or most specific legal provisions.

However, I can suggest that you refer to the BNS 2023 itself ...
Context preview length: 5866
--------------------------------------------------

Q: What is Section 302 of IPC about?
A: Section 302 of the Indian Penal Code (IPC) deals with the punishment for murder.

---
## Part 5: Evaluation (Day 4)

In [30]:
# Load test cases using evaluation.test (reads evaluation/tests.jsonl)
tests = load_tests()
print(f"Loaded {len(tests)} test cases")

# Quick category distribution
categories = {}
for t in tests:
    categories[t.category] = categories.get(t.category, 0) + 1
print(f"Categories: {categories}")

Loaded 10 test cases
Categories: {'direct_fact': 7, 'spanning': 2, 'temporal': 1}


In [31]:
# Evaluate a sample test case
example = tests[0]
print(f"Question: {example.question}")
print(f"Category: {example.category}")
print(f"Reference: {example.reference_answer}")
print(f"Keywords: {example.keywords}")

Question: What is Article 14 of the Indian Constitution about?
Category: direct_fact
Reference: Article 14 of the Indian Constitution guarantees equality before law. It states that the State shall not deny to any person equality before the law or the equal protection of the laws within the territory of India.
Keywords: ['Equality', 'law', 'Article 14']


In [32]:
# Run retrieval evaluation
retrieval_eval = evaluate_retrieval(example)
print(f"Retrieval Evaluation:")
print(f"  MRR: {retrieval_eval.mrr:.3f}")
print(f"  NDCG: {retrieval_eval.ndcg:.3f}")
print(f"  Keyword Coverage: {retrieval_eval.keyword_coverage:.1f}%")

Retrieval Evaluation:
  MRR: 0.222
  NDCG: 0.341
  Keyword Coverage: 66.7%


In [33]:
# Run answer evaluation
answer_eval, answer, chunks = evaluate_answer(example)
print(f"Answer Evaluation:")
print(f"  Accuracy: {answer_eval.accuracy}/5")
print(f"  Completeness: {answer_eval.completeness}/5")
print(f"  Relevance: {answer_eval.relevance}/5")
print(f"  Feedback: {answer_eval.feedback}")

Answer Evaluation:
  Accuracy: 5.0/5
  Completeness: 4.0/5
  Relevance: 5.0/5
  Feedback: Accuracy: 5/5, Completeness: 4/5, Relevance: 5/5


In [34]:
# Batch evaluation using evaluation.eval generators
def run_full_evaluation(max_tests=None):
    retrieval_rows = []
    answer_rows = []

    for i, (test_case, ret_eval, _) in enumerate(evaluate_all_retrieval()):
        if max_tests and i >= max_tests:
            break
        retrieval_rows.append({
            "question": test_case.question,
            "category": test_case.category,
            "mrr": ret_eval.mrr,
            "ndcg": ret_eval.ndcg,
            "keyword_coverage": ret_eval.keyword_coverage,
        })

    for i, (test_case, ans_eval, _) in enumerate(evaluate_all_answers()):
        if max_tests and i >= max_tests:
            break
        answer_rows.append({
            "question": test_case.question,
            "category": test_case.category,
            "accuracy": ans_eval.accuracy,
            "completeness": ans_eval.completeness,
            "relevance": ans_eval.relevance,
        })

    print(f"Retrieved eval rows: {len(retrieval_rows)}")
    print(f"Answer eval rows: {len(answer_rows)}")

    # evaluator.py utility usage example
    print("Sample evaluator color for MRR=0.85:", get_color(0.85, "mrr"))
    return retrieval_rows, answer_rows

# Optional: run dashboard summary functions from evaluator.py
# retrieval_html, retrieval_df = run_retrieval_evaluation()
# answer_html, answer_df = run_answer_evaluation()

# Optional: run quick batch for first 3 tests
# retrieval_rows, answer_rows = run_full_evaluation(max_tests=3)

---
## Part 6: Gradio Chat Interface

In [35]:
# Launch Gradio interface
demo = gr.ChatInterface(
    answer_question,
    title="Legal RAG Assistant",
    description="Ask questions about Indian Constitution, IPC, and BNS",
    examples=[
        "What is Article 21 of the Indian Constitution?",
        "What is Section 302 of IPC?",
        "When was the Bharatiya Nyaya Sanhita enacted?"
    ]
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


---
## Summary

This notebook demonstrates a complete RAG pipeline:

1. **Document Loading**: Load legal documents (COI, IPC, BNS)
2. **Chunking**: Split documents into manageable chunks
3. **Embedding**: Create vector embeddings using HuggingFace
4. **Vector Store**: Store embeddings in Chroma
5. **Retrieval**: Find relevant chunks for a query
6. **Generation**: Use LLM to generate answers
7. **Evaluation**: Measure retrieval and answer quality